# Fair Lending 1B

We'll run code cells here to understand the mortgage decisions made in 2015 for the state of New York.

In [1]:
import pandas as pd
import numpy as np
import os 
import warnings
warnings.filterwarnings('ignore')

In [14]:
mortgage_filename = os.path.join(os.getcwd(), "..", "data", "ny_hmda_2015.csv")
modified_filename = os.path.join(os.getcwd(), "..", "data", "updated_ny_hmda_2015.csv")

df = pd.read_csv(mortgage_filename, header=0)
df_cleaned = pd.read_csv(modified_filename, header=0)
df.head()

,action_taken,action_taken_name,agency_code,agency_abbr,agency_name,applicant_ethnicity,applicant_ethnicity_name,applicant_income_000s,applicant_race_1,applicant_race_2,...,state_abbr,state_name,hud_median_family_income,loan_amount_000s,number_of_1_to_4_family_units,number_of_owner_occupied_units,minority_population,population,rate_spread,tract_to_msamd_income
0,1,Loan originated,9,CFPB,Consumer Financial Protection Bureau,2,Not Hispanic or Latino,97.0,5,NaN,...,NY,New York,109000.0,187,363.0,1817.0,21.139999,5870.0,NaN,109.459999
1,1,Loan originated,9,CFPB,Consumer Financial Protection Bureau,2,Not Hispanic or Latino,200.0,5,NaN,...,NY,New York,71300.0,460,53.0,256.0,45.959999,3512.0,NaN,160.600006
2,1,Loan originated,7,HUD,Department of Housing and Urban Development,2,Not Hispanic or Latino,NaN,3,NaN,...,NY,New York,71300.0,296,2745.0,2586.0,38.990002,8357.0,NaN,134.820007
3,1,Loan originated,9,CFPB,Consumer Financial Protection Bureau,2,Not Hispanic or Latino,202.0,5,NaN,...,NY,New York,109000.0,770,1879.0,2147.0,7.350000,6642.0,NaN,165.830002
4,1,Loan originated,9,CFPB,Consumer Financial Protection Bureau,2,Not Hispanic or Latino,255.0,5,NaN,...,NY,New York,109000.0,648,835.0,676.0,30.059999,2339.0,NaN,133.300003


## Data Understanding

First, let's look at the shape of dataset and the columns.

In [3]:
df.shape

(439654, 78)

In [4]:
df.columns

Index(['action_taken', 'action_taken_name', 'agency_code', 'agency_abbr',
       'agency_name', 'applicant_ethnicity', 'applicant_ethnicity_name',
       'applicant_income_000s', 'applicant_race_1', 'applicant_race_2',
       'applicant_race_3', 'applicant_race_4', 'applicant_race_5',
       'applicant_race_name_1', 'applicant_race_name_2',
       'applicant_race_name_3', 'applicant_race_name_4',
       'applicant_race_name_5', 'applicant_sex', 'applicant_sex_name',
       'application_date_indicator', 'as_of_year', 'census_tract_number',
       'co_applicant_ethnicity', 'co_applicant_ethnicity_name',
       'co_applicant_race_1', 'co_applicant_race_2', 'co_applicant_race_3',
       'co_applicant_race_4', 'co_applicant_race_5',
       'co_applicant_race_name_1', 'co_applicant_race_name_2',
       'co_applicant_race_name_3', 'co_applicant_race_name_4',
       'co_applicant_race_name_5', 'co_applicant_sex', 'co_applicant_sex_name',
       'county_code', 'county_name', 'denial_reason_1', 

The dataset is missing column descriptions, which left me unsure about what some columns represent like "action taken". We'll look through the column values and value counts to determine what these columns are about.

### `action_taken` Target Column

In [5]:
df['action_taken'].value_counts()

action_taken
1    228054
3     79697
6     61490
4     39496
5     16733
2     14180
7         4
Name: count, dtype: int64

It's ambiguous to understand what the numbers represent just by looking at them. We'll need to look at other columns like 'action_taken_name' to understand the column better.

In [6]:
df['action_taken_name'].value_counts()

action_taken_name
Loan originated                                        228054
Application denied by financial institution             79697
Loan purchased by the institution                       61490
Application withdrawn by applicant                      39496
File closed for incompleteness                          16733
Application approved but not accepted                   14180
Preapproval request denied by financial institution         4
Name: count, dtype: int64

Now, we can clearly understand what the numbers from 'action_taken' column represent. Matching the value_counts to each value in the 2 columns, we can conclude that the numbers from 'action_taken' are represented by the descriptions from 'action_taken_name'.

| action_taken | action_taken_name |
| - | - |
| 1 | Loan Originated |
| 2 | Application approved but not accepted |
| 3 | Application denied by financial institution |
| 4 | Application withdrawn by applicant |
| 5 | File closed for incompleteness |
| 6 | Loan purchased by the institution |
| 7 | Preapproval request denied by financial institution |

For our problem, we only want to know if a loan application will be approved or denied, so observing the `action_taken_names` values, we could decide to only include the examples that strongly suggest the approval or denial of a loan application. 

I chose 5 action values that exactly tells me the approval or denial status of a loan application.
- Approval: `Loan Originated`, `Application approved but not accepted`, and `Loan purchased by the institution`
- Denial: `Application denied by financial institution` and `Preapproval request denied by financial institution`

Definitions of the 3 ambiguous action values:
- `Application approved but not accepted`: Application is approved from the financial institution side. It's just the borrower has not accepted the loan yet. This counts as approval from the institution's side, and that's what we care about.
- `Loan purchased by the institution`: The loan was already approved, closed, and funded by an original lender, and a second institution bought the loan afterward. This data would also be relevant to loan approvals.
- `Preapproval request denied by financial institution`: The financial institution denied the loan application with surface level checks like credit scores. This also counts as a denial, even if it was done a little earlier than normal loan requests, where document verification is also done.

To conclude, the target column `is_approved` will be binary values of either 1 or 0 indicating Approval or Denial respectively. 

In order to achieve a binary column, we should
- Remove all examples which are not the 5 chosen action values that indicate approval or denial.
- Combine the 5 chosen action values into 2 binary values of 1 or 0. Approval (1) will have 3 action values, and Denial (0) will have 2 action values.
- Remove columns `action_taken` and `action_taken_name`, as we're moving to binary classification.

### Class Imbalance

### Data Types

Let's first check the number of columns for each data type.

In [17]:
df.dtypes.value_counts()

str        34
float64    23
int64      21
Name: count, dtype: int64

There're 34 string columns, which means one-hot encoding every string column might create lots of columns.

### Missing Values

We'll start by checking if the target `action_taken` column has missing values.

In [7]:
print('Any missing values in `action_taken` column?', df['action_taken'].isnull().values.any())

Any missing values in `action_taken` column? False


`action_taken` column has no missing values, so we can go ahead and shrink it down to a binary column before we handle any missing values in other columns.

In [19]:
df_cleaned['agency_name'].unique()

<StringArray>
[       'Consumer Financial Protection Bureau',
 'Department of Housing and Urban Development',
        'National Credit Union Administration',
       'Federal Deposit Insurance Corporation',
   'Office of the Comptroller of the Currency',
                      'Federal Reserve System']
Length: 6, dtype: str

In [20]:
df_cleaned['agency_code'].unique()

array([9, 7, 5, 3, 1, 2])

In [21]:
df_cleaned['agency_abbr'].unique()

<StringArray>
['CFPB', 'HUD', 'NCUA', 'FDIC', 'OCC', 'FRS']
Length: 6, dtype: str

In [23]:
df_cleaned['applicant_ethnicity_name'].unique()

<StringArray>
[                                                           'Not Hispanic or Latino',
 'Information not provided by applicant in mail, Internet, or telephone application',
                                                                'Hispanic or Latino',
                                                                    'Not applicable']
Length: 4, dtype: str

In [24]:
df_cleaned['applicant_sex_name'].unique()

<StringArray>
[                                                                           'Female',
                                                                              'Male',
 'Information not provided by applicant in mail, Internet, or telephone application',
                                                                    'Not applicable']
Length: 4, dtype: str

In [25]:
df_cleaned['application_date_indicator'].unique()

array([0, 1, 2])

In [29]:
df_cleaned['as_of_year'].dtype

dtype('int64')